# Sanity check on YOLO eports
Once the volunteer boxes have been exported to the YOLO format (via `make_yolo_crops.py`),
you can do some visual verification of the box placements using this notebook.

It selects a random set of jets and plots them all with their corresponding volunteer boxes.

In [ ]:
from PIL import Image
import shapely
import numpy as np
import matplotlib.pyplot as plt
import pathlib
import random

import sunpy.visualization.colormaps as sun_cmaps

In [ ]:
img_path = pathlib.Path('normalized_images')
labels_path = pathlib.Path('labels')

num_samples = 30
selected_images = sorted(random.sample(sorted(img_path.iterdir()), k=num_samples))
selected_labels = sorted(
    labels_path / (ip.stem + '.txt')
    for ip in selected_images
)

In [ ]:
for (img_path, label_path) in zip(selected_images, selected_labels):
    with open(label_path) as f:
        _, bcx, bcy, bw, bh = [float(x) for x in f.read().strip().split()]

    with Image.open(img_path) as f:
        arr = np.asarray(f)
    
    height, width = arr.shape

    # The YOLO values are just proportions of the whole image
    bw *= width
    bh *= height
    bcx *= width
    bcy *= height

    box = shapely.box(
        bcx - bw/2,
        bcy - bh/2,
        bcx + bw/2,
        bcy + bh/2
    )
    # print(box)
    plt.figure()
    plt.imshow(arr, cmap="sdoaia304")
    plt.plot(*box.exterior.xy)
    plt.show()
    plt.close(plt.gcf())